In [28]:
# import os
# os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [29]:
# Import SparkSession to initialize Spark.
from pyspark.sql import SparkSession

In [30]:
spark = SparkSession.builder.appName('Practice_rdd').master('local[*]').getOrCreate()
sc = spark.sparkContext
print(sc)

<SparkContext master=local[*] appName=Practice_rdd>


In [31]:
print(spark.sparkContext.uiWebUrl)

http://10.255.255.254:4040


In [32]:
li = [i for i in range(1,11)]
rdd = sc.parallelize(li)
rdd.getNumPartitions()
rdd.glom().collect()

[[], [1], [], [2], [3], [], [4], [5], [], [6], [], [7], [8], [], [9], [10]]

In [33]:
import csv

file_path = "/root/pyspark/pyspark_practice_2026/real_data_practice/sales.csv"

lines = sc.textFile(file_path)
header = lines.first()

# Parse each CSV line into a list of values, excluding the header.
rows_rdd = (
    lines
    .filter(lambda line: line != header)
    .map(lambda line: next(csv.reader([line])))
)

rows_rdd.take(6)

[['1001', 'C101', 'Bangalore', 'Laptop', '80000'],
 ['1002', 'C102', 'Delhi', 'Phone', '30000'],
 ['1003', 'C101', 'Bangalore', 'Mouse', '2000'],
 ['1004', 'C103', 'Mumbai', 'Laptop', '75000'],
 ['1005', 'C104', 'Delhi', 'Keyboard', '5000'],
 ['1006', 'C101', 'Bangalore', 'Phone', '25000']]

In [34]:
sales = rows_rdd.map(lambda x: tuple((', '.join(x)).split(', ')))
sales.collect()

[('1001', 'C101', 'Bangalore', 'Laptop', '80000'),
 ('1002', 'C102', 'Delhi', 'Phone', '30000'),
 ('1003', 'C101', 'Bangalore', 'Mouse', '2000'),
 ('1004', 'C103', 'Mumbai', 'Laptop', '75000'),
 ('1005', 'C104', 'Delhi', 'Keyboard', '5000'),
 ('1006', 'C101', 'Bangalore', 'Phone', '25000')]

In [35]:
sales = rows_rdd.map(lambda x: (x[0],x[1],x[3]))
sales.collect()

[('1001', 'C101', 'Laptop'),
 ('1002', 'C102', 'Phone'),
 ('1003', 'C101', 'Mouse'),
 ('1004', 'C103', 'Laptop'),
 ('1005', 'C104', 'Keyboard'),
 ('1006', 'C101', 'Phone')]

In [36]:
rows_rdd.map(lambda x:float(x[4]) > 5000).collect()

[True, True, False, True, False, True]

In [37]:
rows_rdd.filter(lambda x:float(x[4]) > 5000).collect()

[['1001', 'C101', 'Bangalore', 'Laptop', '80000'],
 ['1002', 'C102', 'Delhi', 'Phone', '30000'],
 ['1004', 'C103', 'Mumbai', 'Laptop', '75000'],
 ['1006', 'C101', 'Bangalore', 'Phone', '25000']]

In [38]:
rows_rdd.flatMap(lambda x:[x[:4] + [int(x[4])+1000] + x[5:]]).collect()

[['1001', 'C101', 'Bangalore', 'Laptop', 81000],
 ['1002', 'C102', 'Delhi', 'Phone', 31000],
 ['1003', 'C101', 'Bangalore', 'Mouse', 3000],
 ['1004', 'C103', 'Mumbai', 'Laptop', 76000],
 ['1005', 'C104', 'Delhi', 'Keyboard', 6000],
 ['1006', 'C101', 'Bangalore', 'Phone', 26000]]

In [39]:
rdd = sc.textFile('/root/pyspark/pyspark_practice_2026/senior_pyspark_practical_questions.txt')

In [40]:
rdd.union(rows_rdd).collect()

['Senior Data Engineer PySpark Practical Interview Questions',
 '',
 'Files in this folder:',
 '- customers.csv',
 '- orders.csv',
 '- products.json',
 '- events.json',
 '- sales_data.xlsx',
 '',
 '',
 '1. Read customers.csv and compute the count of active customers by country and segment.',
 '',
 '2. Find duplicate customer IDs in the customer master and flag all invalid rows before loading to the warehouse.',
 '',
 '3. Join orders.csv to customers.csv and calculate monthly revenue by state for all completed orders.',
 '',
 '4. Load orders.csv and filter out cancelled and refunded orders before building a daily sales KPI table.',
 '',
 '5. Compute customer lifetime value by summing total_amount for each registered customer and identify top 10 customers.',
 '',
 '6. Use event.json to build a daily funnel from view -> add_to_cart -> purchase and calculate conversion rate.',
 '',
 '7. Read products.json and flatten nested attributes such as screen, ram, and connectivity into separate col

In [41]:
sales = sc.parallelize([
    ("Bangalore", 80000),
    ("Delhi", 30000),
    ("Bangalore", 2000),
    ("Delhi", 5000),
    ("Mumbai", 75000)
])

In [42]:
sales.reduceByKey(lambda x, y: x+y).collect()

[('Delhi', 35000), ('Mumbai', 75000), ('Bangalore', 82000)]

In [43]:
sales.groupByKey().collect()

[('Delhi', <pyspark.resultiterable.ResultIterable at 0x781039b6c170>),
 ('Mumbai', <pyspark.resultiterable.ResultIterable at 0x781039b6df40>),
 ('Bangalore', <pyspark.resultiterable.ResultIterable at 0x781039b6fec0>)]

In [44]:
sales.groupByKey().mapValues(list).collect()

[('Delhi', [30000, 5000]), ('Mumbai', [75000]), ('Bangalore', [80000, 2000])]

In [45]:
sales.groupByKey().mapValues(tuple).collect()

[('Delhi', (30000, 5000)), ('Mumbai', (75000,)), ('Bangalore', (80000, 2000))]

In [46]:
sales.groupBy(lambda x: x[0]).mapValues(list).collect()

[('Delhi', [('Delhi', 30000), ('Delhi', 5000)]),
 ('Mumbai', [('Mumbai', 75000)]),
 ('Bangalore', [('Bangalore', 80000), ('Bangalore', 2000)])]

In [47]:
orders = sc.parallelize([
    (1005, "Keyboard"),
    (1001, "Laptop"),
    (1003, "Mouse"),
    (1002, "Phone")
])

In [48]:
orders.sortByKey().collect()

[(1001, 'Laptop'), (1002, 'Phone'), (1003, 'Mouse'), (1005, 'Keyboard')]

In [49]:
customers = sc.parallelize([
    "C101",
    "C102",
    "C101",
    "C103",
    "C102"
])

In [50]:
customers.distinct().collect()

['C102', 'C103', 'C101']

In [51]:
customers = sc.parallelize([
    ("C101", "Basu"),
    ("C102", "Rahul"),
    ("C103", "Amit")
])

In [52]:
sales = sc.parallelize([
    ("C101", 80000),
    ("C102", 30000),
    ("C101", 2000)
])

In [53]:
sales.join(customers).collect()

[('C101', (80000, 'Basu')),
 ('C101', (2000, 'Basu')),
 ('C102', (30000, 'Rahul'))]

In [54]:
sales.getNumPartitions()

16

In [55]:
sales = sales.repartition(20)

In [56]:
sales.glom().collect()

[[],
 [],
 [],
 [],
 [],
 [],
 [('C101', 80000)],
 [],
 [],
 [],
 [('C102', 30000)],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [('C101', 2000)]]

In [57]:
sales = sales.coalesce(2)
sales.glom().collect()

[[('C101', 80000)], [('C101', 2000), ('C102', 30000)]]

In [58]:
sales.count()

3

In [59]:
sales.first()

('C101', 80000)

In [60]:
sales.take(2)

[('C101', 80000), ('C101', 2000)]

In [61]:
sales.top(2)

[('C102', 30000), ('C101', 80000)]

In [62]:
amounts = sc.parallelize([100, 200, 300, 400])

In [63]:
total = amounts.reduce(
    lambda a, b: a + b
)

In [64]:
print(total)

1000


In [65]:
total = amounts.fold(
    0,
    lambda a, b: a + b
)

In [66]:
print(total)

1000


In [67]:
result = amounts.aggregate(
    (0, 0),
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda acc1, acc2: (
        acc1[0] + acc2[0],
        acc1[1] + acc2[1]
    )
)

In [68]:
print(result)

(1000, 4)


In [69]:
sales.foreach(lambda x:print(x))

('C101', 2000)
('C102', 30000)
('C101', 80000)


In [70]:
#sales.saveAsTextFile('/root/pyspark/pyspark_practice_2026/test.txt')

In [71]:
spark.stop()